# TRAITEMENT DE DONNÉES

# Modélisation des causes et les caractéristiques des accidents pour réduire leur nombre et leur gravité

En statistique et en Machine Learning, la variable cible (ou target) est celle que l'on cherche à prédire ou à expliquer en fonction de toutes les autres (les variables explicatives). Donc La gravité **grav** est la variable idéale pour expliquer les facteurs liés à l'accident survenu.

C'est sur cette variable que se basent les politiques publiques.

- Si mon modèle va montrer que la combinaison "Pluie" (atm) + "Route départementale" (catr), etc... augmente drastiquement la gravité, l'État peut décider d'abaisser la vitesse maximale (vma) sur ces zones précises.

## Importation des librairies

In [10]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split

## Importation des bases de données

In [6]:
def nettoyer_colonnes(df):

    """

    Nettoie les noms de colonnes d'un DataFrame.
 
    Transformations appliquées :

    - mise en minuscules

    - remplacement des points (.) par des underscores (_)

    """
 
    # Nettoyage des noms de colonnes

    df.columns = (

        df.columns

            .str.lower()                     # tout en minuscules

            .str.replace('.', '_')           # remplace les points par _

    )
 
    return df

* **Base caracteristique**

In [3]:
df_caract = pd.read_csv("caracteristique.txt", sep="\t")  # sep="," si CSV classique
df_caract.head(10)

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,col,adr,lat,long
0,202100037733,28,5,2021,14:15,1,CC_97407,2,2,1,3,JEAN MOULIN (RUE) - RIV. DES GALETS,"-20,95888","55,31743"
1,202100049382,2,3,2021,10:59,1,CC_63113,2,1,1,2,SAINT ROBERT (RUE),"45,79137","3,1162"
2,202100038891,20,5,2021,16:50,1,CC_88160,2,2,1,1,POISSOMPRE (FG DE),"48,18261","6,47218"
3,202100028616,14,7,2021,19:45,1,CC_80021,2,1,1,3,CAGNARD (RUE),"49,90355","2,2872"
4,202100033509,17,6,2021,23:40,3,CC_13214,2,1,1,1,A7-Sens Lyon vers Marseille,"43,34031","5,37689"
5,202100048274,12,3,2021,16:25,1,CC_30189,2,2,1,6,TALABOT (BOULEVARD),"43,83843726","4,372475088"
6,202100054416,18,1,2021,15:25,1,CC_91228,2,3,1,2,BD CHAMPS ELYSEES,"48,633158","2,425203"
7,202100026295,29,7,2021,17:35,1,CC_51230,2,1,1,7,Rue Pierre Semard,"49,04566","3,96145"
8,202100010947,26,10,2021,09:10,1,CC_93059,2,3,1,3,MERCHER (PASSAGE),"48,94933","2,35775"
9,202100009689,4,11,2021,08:35,1,CC_55369,1,1,5,6,D28,"48,808277","5,211725"


In [4]:
# Cette commande détecte automatiquement si une colonne doit être int, string, objet ou boolean
df_caract = df_caract.convert_dtypes()

In [5]:
def explore_base(df_caract):
    print("\033[1mFormat\033[0m")
    print("----------------------------------")
    print(df_caract.info())
    print("----------------------------------")
    print("\033[1mContrôle des valeurs manquantes\033[0m")
    print("----------------------------------")
    print(df_caract.isna().sum().sort_values(ascending=False))

    for i in df_caract.columns:
        if df_caract[i].dtypes in ['int64','float64']:
            print("----------------------------------")
            print("\033[1m Variable :\033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_caract[i].describe())
            print("----------------------------------")
            print("\033[1m Modalité \033[0m")
            print("----------------------------------\n")
            print(df_caract[i].unique())

        else:
            print("----------------------------------")
            print("\033[1m Variable : \033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_caract[i].value_counts(dropna=False))

In [13]:
explore_base(df_caract)

Format
----------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56530 entries, 0 to 56529
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Num_Acc  56530 non-null  int64 
 1   jour     56530 non-null  int64 
 2   mois     56530 non-null  int64 
 3   an       56530 non-null  int64 
 4   hrmn     56530 non-null  object
 5   lum      56530 non-null  int64 
 6   com      56530 non-null  object
 7   agg      56530 non-null  int64 
 8   int      56530 non-null  int64 
 9   atm      56530 non-null  int64 
 10  col      56530 non-null  int64 
 11  adr      55957 non-null  object
 12  lat      56530 non-null  object
 13  long     56530 non-null  object
dtypes: int64(9), object(5)
memory usage: 6.0+ MB
None
----------------------------------
Contrôle des valeurs manquantes
----------------------------------
adr        573
Num_Acc      0
jour         0
mois         0
an           0
hrmn         0
lum       

* **Base Lieux**

In [33]:
df_lieux = pd.read_excel("lieux.xls")
df_lieux.head(10)

,Num_Acc,dep,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
0,202100022009,40,1,A63,0,A,3,3,0,1,141,600,1,NaN,-1.0,1,0,1,130
1,202100023746,94,4,VLADIMIR ILLITCH LENINE (AV),0,NaN,-1,2,0,1,0,0,1,NaN,-1.0,1,0,1,50
2,202100019549,2A,2,20,0,T,2,2,0,1,-1,-1,2,NaN,-1.0,1,0,3,80
3,202100022748,75,4,AVENUE SIMON BOLIVAR,0,NaN,2,2,0,2,-1,-1,1,NaN,-1.0,1,0,1,50
4,202100051993,971,3,BOLOGNE (ROUTE DE),0,NaN,2,2,0,1,-1,-1,1,NaN,-1.0,1,0,1,50
5,202100038809,57,3,63,0,A,2,2,0,1,-1,170,1,NaN,-1.0,2,0,1,50
6,202100023370,971,4,NaN,0,NaN,2,1,1,1,-1,-1,1,NaN,3.0,1,0,3,50
7,202100038829,17,3,939,-1,NaN,2,2,0,1,28,659,1,NaN,-1.0,2,0,1,80
8,202100013461,66,3,916,0,NaN,2,2,0,1,6,645,1,NaN,-1.0,1,0,1,70
9,202100017993,78,3,GENERAL DE GAULLE (RUE DU),0,NaN,-1,0,0,1,38,662,1,NaN,-1.0,1,9,1,50


In [10]:
# Cette commande détecte automatiquement si une colonne doit être int, string, objet ou boolean
df_lieux = df_lieux.convert_dtypes()

In [38]:
def explore_base(df_lieux):
    print("\033[1mFormat\033[0m")
    print("----------------------------------")
    print(df_lieux.info())
    print("----------------------------------")
    print("\033[1mContrôle des valeurs manquantes\033[0m")
    print("----------------------------------")
    print(df_lieux.isna().sum().sort_values(ascending=False))

    for i in df_lieux.columns:
        if df_lieux[i].dtypes in ['int64','float64']:
            print("----------------------------------")
            print("\033[1m Variable :\033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_lieux[i].describe())
            print("----------------------------------")
            print("\033[1m Modalité \033[0m")
            print("----------------------------------\n")
            print(df_lieux[i].unique())

        else:
            print("----------------------------------")
            print("\033[1m Variable : \033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_lieux[i].value_counts(dropna=False))

In [39]:
explore_base(df_lieux)

Format
----------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56525 entries, 0 to 56524
Data columns (total 19 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Num_Acc  56525 non-null  int64  
 1   dep      56525 non-null  object 
 2   catr     56525 non-null  int64  
 3   voie     52117 non-null  object 
 4   v1       56525 non-null  int64  
 5   v2       4990 non-null   object 
 6   circ     56525 non-null  int64  
 7   nbv      56525 non-null  int64  
 8   vosp     56525 non-null  int64  
 9   prof     56525 non-null  int64  
 10  pr       56525 non-null  int64  
 11  pr1      56525 non-null  int64  
 12  plan     56525 non-null  int64  
 13  lartpc   108 non-null    float64
 14  larrout  56525 non-null  float64
 15  surf     56525 non-null  int64  
 16  infra    56525 non-null  int64  
 17  situ     56525 non-null  int64  
 18  vma      56525 non-null  int64  
dtypes: float64(2), int64(14), object(3)
memory usa

* **Base usagers**

In [135]:
df_usag = pd.read_csv("usagers.csv")
df_usag.head(10)

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202100021165,163Â 830,A01,9,2,4,1,1948.0,0,1,-1,-1,0,0,-1
1,202100020281,165Â 383,A01,1,1,1,2,1965.0,1,1,8,-1,-1,-1,-1
2,202100047336,117Â 153,A01,1,1,4,2,1972.0,0,2,0,-1,-1,-1,-1
3,202100008918,185Â 779,B01,1,1,4,1,1985.0,9,8,0,-1,-1,-1,-1
4,202100009175,185Â 315,A01,1,1,3,1,1960.0,5,1,-1,-1,0,0,-1
5,202100050277,111Â 849,B01,1,1,4,1,1972.0,3,1,9,-1,0,0,-1
6,202100027012,153Â 433,A01,1,1,1,1,1986.0,5,1,-1,-1,0,0,-1
7,202100028064,151Â 549,A01,1,1,1,2,1957.0,5,1,-1,-1,0,0,-1
8,202100003701,195Â 137,A01,1,1,4,2,2006.0,0,2,0,-1,-1,-1,-1
9,202100033813,141Â 320,A01,1,1,1,1,1999.0,5,1,8,-1,-1,-1,-1


In [136]:
df_usag.shape

(129189, 15)

In [137]:
df_usag.isna().sum()

Num_Acc           0
id_vehicule       0
num_veh           0
place             0
catu              0
grav              0
sexe              0
an_nais        3067
trajet            0
secu1             0
secu2             0
secu3             0
locp              0
actp              0
etatp             0
dtype: int64

In [145]:
df_usag[df_usag['an_nais'].isna()].sort_values('id_vehicule')

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
127037,202100056500,100Â 909,A01,1,1,1,-1,NaN,-1,-1,-1,-1,-1,-1,-1
18360,202100056487,100Â 930,Z01,1,1,1,-1,NaN,-1,0,0,0,-1,-1,-1
25321,202100056485,100Â 935,Z01,1,1,1,-1,NaN,-1,-1,-1,-1,-1,-1,-1
66525,202100056472,100Â 959,B01,1,1,1,-1,NaN,-1,-1,-1,-1,-1,-1,-1
8253,202100056439,101Â 015,A01,1,1,1,-1,NaN,-1,-1,-1,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93622,202100000051,201Â 657,B01,1,1,1,-1,NaN,-1,-1,-1,-1,-1,-1,-1
8488,202100000045,201Â 676,A01,1,1,1,-1,NaN,-1,-1,-1,-1,-1,-1,-1
48815,202100000038,201Â 694,A01,1,1,1,-1,NaN,-1,8,-1,-1,0,0,-1
52373,202100000020,201Â 725,A01,1,1,1,-1,NaN,-1,-1,-1,-1,-1,-1,-1


In [150]:
df_usag.dropna(subset=['an_nais'], inplace=True)

In [173]:
#df_usag[df_usag['sexe']==-1].sort_values('id_vehicule')

In [172]:
df_usag[df_usag['sexe'].isin([0, -1])]

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
49978,202100021078,163A 980,B01,1,1,1,-1,1993,-1,8,8,8,0,0,-1
56737,202100034601,139A 865,A01,1,1,1,-1,1992,-1,8,8,8,0,0,-1


In [174]:
df_usag = df_usag[~df_usag['sexe'].isin([0, -1])]

In [176]:
# 1) Corriger les caractères dans id_vehicule
df_usag["id_vehicule"] = df_usag["id_vehicule"].str.replace("Â", "A", regex=False)

# 2) Supprimer le .0 dans an_nais (ex: 1948.0 → 1948)
df_usag["an_nais"] = df_usag["an_nais"].astype("Int64")
df_usag.head(10)

/var/folders/yp/sy_lpvd138s8qtlhb_yrqtjm0000gn/T/ipykernel_14118/1899422222.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_usag["id_vehicule"] = df_usag["id_vehicule"].str.replace("Â", "A", regex=False)
/var/folders/yp/sy_lpvd138s8qtlhb_yrqtjm0000gn/T/ipykernel_14118/1899422222.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_usag["an_nais"] = df_usag["an_nais"].astype("Int64")


,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202100021165,163A 830,A01,9,2,4,1,1948,0,1,-1,-1,0,0,-1
1,202100020281,165A 383,A01,1,1,1,2,1965,1,1,8,-1,-1,-1,-1
2,202100047336,117A 153,A01,1,1,4,2,1972,0,2,0,-1,-1,-1,-1
3,202100008918,185A 779,B01,1,1,4,1,1985,9,8,0,-1,-1,-1,-1
4,202100009175,185A 315,A01,1,1,3,1,1960,5,1,-1,-1,0,0,-1
5,202100050277,111A 849,B01,1,1,4,1,1972,3,1,9,-1,0,0,-1
6,202100027012,153A 433,A01,1,1,1,1,1986,5,1,-1,-1,0,0,-1
7,202100028064,151A 549,A01,1,1,1,2,1957,5,1,-1,-1,0,0,-1
8,202100003701,195A 137,A01,1,1,4,2,2006,0,2,0,-1,-1,-1,-1
9,202100033813,141A 320,A01,1,1,1,1,1999,5,1,8,-1,-1,-1,-1


In [178]:
# Cette commande détecte automatiquement si une colonne doit être int, string, objet ou boolean
df_usag = df_usag.convert_dtypes()

In [179]:
def explore_base(df_usag):
    print("\033[1mFormat\033[0m")
    print("----------------------------------")
    print(df_usag.info())
    print("----------------------------------")
    print("\033[1mContrôle des valeurs manquantes\033[0m")
    print("----------------------------------")
    print(df_usag.isna().sum().sort_values(ascending=False))

    for i in df_usag.columns:
        if df_usag[i].dtypes in ['int64','float64']:
            print("----------------------------------")
            print("\033[1m Variable :\033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_usag[i].describe())
            print("----------------------------------")
            print("\033[1m Modalité \033[0m")
            print("----------------------------------\n")
            print(df_usag[i].unique())

        else:
            print("----------------------------------")
            print("\033[1m Variable : \033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_usag[i].value_counts(dropna=False))

In [180]:
explore_base(df_usag)

Format
----------------------------------
<class 'pandas.core.frame.DataFrame'>
Index: 126120 entries, 0 to 129188
Data columns (total 15 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Num_Acc      126120 non-null  Int64 
 1   id_vehicule  126120 non-null  string
 2   num_veh      126120 non-null  string
 3   place        126120 non-null  Int64 
 4   catu         126120 non-null  Int64 
 5   grav         126120 non-null  Int64 
 6   sexe         126120 non-null  Int64 
 7   an_nais      126120 non-null  Int64 
 8   trajet       126120 non-null  Int64 
 9   secu1        126120 non-null  Int64 
 10  secu2        126120 non-null  Int64 
 11  secu3        126120 non-null  Int64 
 12  locp         126120 non-null  Int64 
 13  actp         126120 non-null  string
 14  etatp        126120 non-null  Int64 
dtypes: Int64(12), string(3)
memory usage: 16.8 MB
None
----------------------------------
Contrôle des valeurs manquantes
---------------

* **Base vehicules**

In [129]:
df_veh = pd.read_csv("vehicules.csv",sep=";", encoding="latin-1", decimal=",")
df_veh.head(10)

,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
0,129åÊ983,A01,1,33,0,2,3,1,1,NaN
1,192åÊ850,B01,1,10,0,2,4,2,1,NaN
2,152åÊ776,A01,1,33,0,2,1,3,3,NaN
3,193åÊ398,A01,1,7,0,1,2,1,1,NaN
4,178åÊ605,A01,2,7,0,1,2,1,1,NaN
5,187åÊ823,A01,2,30,0,2,1,1,1,NaN
6,103åÊ719,B01,1,1,0,2,1,1,5,NaN
7,164åÊ572,B01,1,7,8,2,2,1,1,NaN
8,190åÊ438,B01,3,15,0,2,3,1,1,NaN
9,146åÊ391,B01,1,7,0,2,4,2,1,NaN


In [131]:
df_veh.shape

(97365, 10)

In [73]:
# Modification format id_vehicule

In [74]:
df_veh["id_vehicule"]=df_veh["id_vehicule"].str.replace("åÊ", "A", regex=False)
df_veh.head()

,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
0,129A983,A01,1,33,0,2,3,1,1,NaN
1,192A850,B01,1,10,0,2,4,2,1,NaN
2,152A776,A01,1,33,0,2,1,3,3,NaN
3,193A398,A01,1,7,0,1,2,1,1,NaN
4,178A605,A01,2,7,0,1,2,1,1,NaN


**Exploration de la base vehicule**

In [75]:
# Cette commande détecte automatiquement si une colonne doit être int, string, objet ou boolean
df_veh = df_veh.convert_dtypes()

In [76]:
def explore_base(df_veh):
    print("\033[1mFormat\033[0m")
    print("----------------------------------")
    print(df_veh.info())
    print("----------------------------------")
    print("\033[1mContrôle des valeurs manquantes\033[0m")
    print("----------------------------------")
    print(df_veh.isna().sum().sort_values(ascending=False))

    for i in df_veh.columns:
        if df_veh[i].dtypes in ['int64','float64']:
            print("----------------------------------")
            print("\033[1m Variable :\033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_veh[i].describe())
            print("----------------------------------")
            print("\033[1m Modalité \033[0m")
            print("----------------------------------\n")
            print(df_veh[i].unique())

        else:
            print("----------------------------------")
            print("\033[1m Variable : \033[0m",i)
            print("----------------------------------")
            print("\033[1m Description \033[0m")
            print("----------------------------------\n")
            print(df_veh[i].value_counts(dropna=False))

In [77]:
explore_base(df_veh)

Format
----------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97365 entries, 0 to 97364
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id_vehicule  97365 non-null  string
 1   num_veh      97365 non-null  string
 2   senc         97365 non-null  Int64 
 3   catv         97365 non-null  Int64 
 4   obs          97365 non-null  Int64 
 5   obsm         97365 non-null  Int64 
 6   choc         97365 non-null  Int64 
 7   manv         97365 non-null  Int64 
 8   motor        97365 non-null  Int64 
 9   occutc       744 non-null    Int64 
dtypes: Int64(8), string(2)
memory usage: 8.2 MB
None
----------------------------------
Contrôle des valeurs manquantes
----------------------------------
occutc         96621
id_vehicule        0
num_veh            0
senc               0
catv               0
obs                0
obsm               0
choc               0
manv               0
motor       

## Exporter les bases propres 

In [132]:
# Export des 4 bases dans le dossier 'data'
df_caract.to_csv('data/caracteristiques_clean.csv', index=False, sep=';', encoding='utf-8-sig')
df_lieux.to_csv('data/lieux_clean.csv', index=False, sep=';', encoding='utf-8-sig')
df_usag_clean.to_csv('data/usagers_clean.csv', index=False, sep=';', encoding='utf-8-sig')
df_veh_clean.to_csv('data/vehicules_clean.csv', index=False, sep=';', encoding='utf-8-sig')

# la base finale fusionnée
df_final.to_csv('data/base_complete.csv', index=False, sep=';', encoding='utf-8-sig')

print("Félicitations ! Tes 5 fichiers sont bien rangés dans le dossier /data.")

Félicitations ! Tes 5 fichiers sont bien rangés dans le dossier /data.


## Vérification des doublon et nettoyage des données dans chaque base

* **Base caracteristique**

**Traitement des doublons pures et impures**

In [88]:
df_caract.duplicated().sum()

np.int64(11)

In [89]:
#Doublons pure

In [90]:
df_caract[df_caract.duplicated(keep=False)].sort_values("Num_Acc")

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,col,adr,lat,long
33687,202100013498,12,10,2021,11:00,1,CC_71076,2,2,1,3,Avenue de Paris,"46,79438402","4,848719001"
6317,202100013498,12,10,2021,11:00,1,CC_71076,2,2,1,3,Avenue de Paris,"46,79438402","4,848719001"
25206,202100017917,21,9,2021,09:45,1,CC_95280,2,1,1,3,RD 47A,"49,01144865","2,479454682"
19025,202100017917,21,9,2021,09:45,1,CC_95280,2,1,1,3,RD 47A,"49,01144865","2,479454682"
32639,202100023406,19,8,2021,10:00,1,CC_97120,2,1,1,6,UNIVERSITE FOUILLOLE,"16,22706","-61,52962"
420,202100023406,19,8,2021,10:00,1,CC_97120,2,1,1,6,UNIVERSITE FOUILLOLE,"16,22706","-61,52962"
45121,202100027321,23,7,2021,11:10,1,CC_97302,2,1,1,3,A.HENRI(RUE),"4,922661236","-52,3113595"
50662,202100027321,23,7,2021,11:10,1,CC_97302,2,1,1,3,A.HENRI(RUE),"4,922661236","-52,3113595"
2994,202100027995,20,7,2021,18:20,1,CC_62307,2,1,1,1,Route de Peuplingues,"50,914111","1,724478"
55470,202100027995,20,7,2021,18:20,1,CC_62307,2,1,1,1,Route de Peuplingues,"50,914111","1,724478"


In [91]:
#Doublons inpures

In [92]:
df_caract[df_caract.duplicated(subset=["Num_Acc"], keep=False)].sort_values("Num_Acc")

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,col,adr,lat,long
33687,202100013498,12,10,2021,11:00,1,CC_71076,2,2,1,3,Avenue de Paris,"46,79438402","4,848719001"
6317,202100013498,12,10,2021,11:00,1,CC_71076,2,2,1,3,Avenue de Paris,"46,79438402","4,848719001"
25206,202100017917,21,9,2021,09:45,1,CC_95280,2,1,1,3,RD 47A,"49,01144865","2,479454682"
19025,202100017917,21,9,2021,09:45,1,CC_95280,2,1,1,3,RD 47A,"49,01144865","2,479454682"
32639,202100023406,19,8,2021,10:00,1,CC_97120,2,1,1,6,UNIVERSITE FOUILLOLE,"16,22706","-61,52962"
420,202100023406,19,8,2021,10:00,1,CC_97120,2,1,1,6,UNIVERSITE FOUILLOLE,"16,22706","-61,52962"
45121,202100027321,23,7,2021,11:10,1,CC_97302,2,1,1,3,A.HENRI(RUE),"4,922661236","-52,3113595"
50662,202100027321,23,7,2021,11:10,1,CC_97302,2,1,1,3,A.HENRI(RUE),"4,922661236","-52,3113595"
2994,202100027995,20,7,2021,18:20,1,CC_62307,2,1,1,1,Route de Peuplingues,"50,914111","1,724478"
55470,202100027995,20,7,2021,18:20,1,CC_62307,2,1,1,1,Route de Peuplingues,"50,914111","1,724478"


**Commentaire :**  Dans la base caracteristique, je constate la présence de **11** doublons

In [93]:
# Supprime les doublons en gardant la première occurrence
df_caract = df_caract.drop_duplicates(subset=["Num_Acc"], keep='first')

In [94]:
df_caract[df_caract.duplicated(subset=["Num_Acc"], keep=False)].sort_values("Num_Acc")

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,col,adr,lat,long


In [95]:
df_caract[df_caract.duplicated(keep=False)].sort_values("Num_Acc")

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,col,adr,lat,long


In [96]:
df_caract.duplicated().sum()

np.int64(0)

In [97]:
print(df_caract.Num_Acc.nunique())
#print(df_caract.shape)

56519


In [98]:
df_caract_clean=df_caract
df_caract_clean

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,col,adr,lat,long
0,202100037733,28,5,2021,14:15,1,CC_97407,2,2,1,3,JEAN MOULIN (RUE) - RIV. DES GALETS,"-20,95888","55,31743"
1,202100049382,2,3,2021,10:59,1,CC_63113,2,1,1,2,SAINT ROBERT (RUE),"45,79137","3,1162"
2,202100038891,20,5,2021,16:50,1,CC_88160,2,2,1,1,POISSOMPRE (FG DE),"48,18261","6,47218"
3,202100028616,14,7,2021,19:45,1,CC_80021,2,1,1,3,CAGNARD (RUE),"49,90355","2,2872"
4,202100033509,17,6,2021,23:40,3,CC_13214,2,1,1,1,A7-Sens Lyon vers Marseille,"43,34031","5,37689"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56525,202100026753,28,7,2021,17:10,1,CC_38474,2,2,1,3,Rue du Vinay,"45,204957","5,667561"
56526,202100019814,10,9,2021,06:15,3,CC_77251,1,1,1,7,RN 104,"48,631","2,533"
56527,202100016048,29,9,2021,08:15,1,CC_78524,1,1,1,3,A13,"48,841059","2,106997"
56528,202100039830,15,5,2021,06:45,2,CC_13212,2,2,1,3,PIERRE CHEVALIER (avenue),"43,2953","5,4254"


* **Base lieux**

In [99]:
df_lieux.duplicated().sum()

np.int64(0)

**Commentaire :**  Dans la base lieux, je constate la présence de **0** doublons

**Remplacer *v2* par une valeur par défaut (Imputation)**

Pour les colonnes textuelles comme voie tel que **v2** qui est la lettre de l'indice alphanumérique de la route est importante, donc j'ai remplacé par "Inconnu"

In [100]:
df_lieux["v2"] = df_lieux["v2"].fillna("Inconnu")
df_lieux["v2"]

0              A
1        Inconnu
2              T
3        Inconnu
4        Inconnu
          ...   
56520    Inconnu
56521    Inconnu
56522    Inconnu
56523    Inconnu
56524    Inconnu
Name: v2, Length: 56525, dtype: object

In [101]:
print(df_lieux["v2"].value_counts())

v2
Inconnu    51535
D           3164
A            896
N            225
C            119
E            106
R             92
B             85
M             74
 -            67
T             43
F             24
X             15
V             14
O             12
U              9
G              8
S              7
Z              6
L              5
P              5
H              3
K              3
J              3
W              2
I              2
34             1
Name: count, dtype: int64


**Gérer les NaN de la colonnes lartpc**

si **lartpc** est la largeur du terre-plein central (TPC) s'il existe (en m), contient plus de 80% de **NaN**, elle n'apporte aucune valeur statistique. Donc je l'ai supprimé

In [104]:
df_lieux.columns

Index(['Num_Acc', 'dep', 'catr', 'voie', 'v1', 'v2', 'circ', 'nbv', 'vosp',
       'prof', 'pr', 'pr1', 'plan', 'larrout', 'surf', 'infra', 'situ', 'vma'],
      dtype='object')

In [105]:
# Calcul du pourcentage de NaN pour lartpc
#pct_nan_lartpc = df_lieux['lartpc'].isnull().mean() * 100

#print(f"Le pourcentage de valeurs manquantes dans 'lartpc' est de : {pct_nan_lartpc:.2f}%")

In [ ]:
# Supprime les colonnes qui ont trop de vides
limit = len(df_lieux) * 0.5  # Seuil de 50%
df_lieux = df_lieux.dropna(thresh=limit, axis=1)

In [106]:
df_lieux_clean=df_lieux
df_lieux_clean

,Num_Acc,dep,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,larrout,surf,infra,situ,vma
0,202100022009,40,1,A63,0,A,3,3,0,1,141,600,1,-1.0,1,0,1,130
1,202100023746,94,4,VLADIMIR ILLITCH LENINE (AV),0,Inconnu,-1,2,0,1,0,0,1,-1.0,1,0,1,50
2,202100019549,2A,2,20,0,T,2,2,0,1,-1,-1,2,-1.0,1,0,3,80
3,202100022748,75,4,AVENUE SIMON BOLIVAR,0,Inconnu,2,2,0,2,-1,-1,1,-1.0,1,0,1,50
4,202100051993,971,3,BOLOGNE (ROUTE DE),0,Inconnu,2,2,0,1,-1,-1,1,-1.0,1,0,1,50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56520,202100056035,76,4,LYONS. ROUTE DE (BRETELLE C),0,Inconnu,1,2,0,1,-1,-1,1,-1.0,1,-1,1,30
56521,202100043915,75,4,RUE LOUIS BLANC,0,Inconnu,1,2,0,1,-1,-1,1,-1.0,1,0,1,30
56522,202100047332,73,4,JEUX OLYMPIQUES (AVENUE DES),0,Inconnu,3,2,2,1,-1,-1,1,-1.0,1,-1,1,50
56523,202100016135,64,4,SERGENT BERNES CAMBOT DU,0,Inconnu,2,3,1,1,0,0,1,-1.0,1,0,1,30


* **Base usagers**

**Traitement des doublons pures et impures**

In [107]:
df_usag.duplicated().sum()

np.int64(33)

**Commentaire :** Dans la base usagers, je constate la présence de 33 doublons

In [108]:
#Doublons pures
df_usag[df_usag.duplicated(keep=False)].sort_values("Num_Acc")

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
72758,202100001471,199A 156,B01,1,1,1,1,1995,9,1,0,-1,-1,-1,-1
55476,202100001471,199A 156,B01,1,1,1,1,1995,9,1,0,-1,-1,-1,-1
28399,202100003260,195A 936,A01,10,3,4,2,2002,5,0,-1,-1,3,3,2
108756,202100003260,195A 936,A01,10,3,4,2,2002,5,0,-1,-1,3,3,2
77041,202100006907,189A 424,A01,2,2,4,1,1998,5,1,0,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78604,202100050391,111A 622,B01,1,1,1,1,1975,4,1,-1,-1,0,0,-1
118307,202100052378,108A 090,B01,1,1,4,1,2001,0,2,0,-1,-1,-1,-1
48087,202100052378,108A 090,B01,1,1,4,1,2001,0,2,0,-1,-1,-1,-1
9691,202100052486,107A 904,A01,1,1,3,1,1990,5,1,-1,-1,0,0,-1


In [109]:
# Supprime les doublons en gardant la première occurrence
df_usag = df_usag.drop_duplicates(subset=["Num_Acc"], keep='first')
df_usag.duplicated().sum()

np.int64(0)

In [110]:
df_usag[df_usag.duplicated(keep=False)].sort_values("Num_Acc")

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp


In [111]:
#df_usag.copy()

In [112]:
#Doublons inpures
df_usag[df_usag.duplicated(subset=["Num_Acc"], keep=False)].sort_values("Num_Acc")

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp


In [113]:
# Pour voir s'il y a des vraies erreurs (même accident, même véhicule, même place)
df_usag[df_usag.duplicated(subset=["Num_Acc", "num_veh", "id_vehicule", "place", "an_nais", "sexe"], keep=False)]

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp


In [128]:
print(df_usag.id_vehicule.nunique())
print(df_usag.shape)

56518
(56521, 15)


In [115]:
df_usag_clean=df_usag
df_usag_clean

,Num_Acc,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202100021165,163A 830,A01,9,2,4,1,1948,0,1,-1,-1,0,0,-1
1,202100020281,165A 383,A01,1,1,1,2,1965,1,1,8,-1,-1,-1,-1
2,202100047336,117A 153,A01,1,1,4,2,1972,0,2,0,-1,-1,-1,-1
3,202100008918,185A 779,B01,1,1,4,1,1985,9,8,0,-1,-1,-1,-1
4,202100009175,185A 315,A01,1,1,3,1,1960,5,1,-1,-1,0,0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129149,202100055940,101A 892,A01,1,1,4,1,1989,1,2,6,-1,0,0,-1
129173,202100054777,103A 879,A01,1,1,2,2,1935,5,1,0,-1,0,0,-1
129186,202100056519,163A 830,A01,9,2,4,1,1948,0,1,-1,-1,0,0,-1
129187,202100056520,165A 383,A01,1,1,1,2,1965,1,1,8,-1,-1,-1,-1


* **Base vehicules**

In [116]:
df_veh.duplicated().sum()

np.int64(0)

**Commentaire :**  Dans la base vehicule, je constate la présence de **0** doublons

In [117]:
# Vérifier s'il y a au moins une valeur remplie
print(df_veh['occutc'].notnull().sum())

744


In [118]:
# Remplacer les NaN par 0
df_veh['occutc'] = df_veh['occutc'].fillna(0)

# Convertir en entier (pour enlever le .0)
df_veh['occutc'] = df_veh['occutc'].astype(int)

In [119]:
df_veh_clean=df_veh
df_veh_clean

,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
0,129A983,A01,1,33,0,2,3,1,1,0
1,192A850,B01,1,10,0,2,4,2,1,0
2,152A776,A01,1,33,0,2,1,3,3,0
3,193A398,A01,1,7,0,1,2,1,1,0
4,178A605,A01,2,7,0,1,2,1,1,0
...,...,...,...,...,...,...,...,...,...,...
97360,108A341,A01,1,1,0,0,0,1,5,0
97361,200A693,B01,1,1,0,2,5,26,5,0
97362,192A529,B01,1,31,0,2,1,1,1,0
97363,174A733,B01,1,2,0,2,3,1,1,0


In [120]:
df_veh[df_veh['occutc']!=0]

,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
74,172A370,A01,1,37,0,2,3,0,1,1
125,145A258,B01,1,37,0,2,6,1,1,1
224,154A771,B01,1,37,0,2,7,1,1,1
262,178A037,B01,2,37,0,2,4,1,1,1
278,110A314,B01,2,38,0,2,3,1,1,1
...,...,...,...,...,...,...,...,...,...,...
97025,130A817,B01,2,37,0,2,8,0,1,1
97112,151A326,A01,1,37,0,2,3,19,1,4
97150,155A428,B01,1,37,0,0,8,15,1,1
97184,179A775,B01,2,37,0,2,4,23,6,1


**Test de correlation**

In [121]:
#tableau de contingence(tableau croisé)
tab=pd.crosstab(df_veh_clean.id_vehicule,df_veh_clean.senc) #tableau de contingence(tableau croisé)
tab

senc,-1,0,1,2,3
id_vehicule,,,,,
100A882,0,0,0,1,0
100A883,0,0,1,0,0
100A884,0,0,0,1,0
100A885,0,0,1,0,0
100A886,0,0,0,1,0
...,...,...,...,...,...
206A069,0,0,1,0,0
206A070,0,0,1,0,0
206A071,0,0,1,0,0


In [122]:
# P-value de test de fisher
#oddsratio, p_value = fisher_exact(tab)
#oddsratio, p_value

In [123]:
#test de Khi-deux
#chi2, p_value, dof, expectid= chi2_contingency(tab)

In [124]:
#Résultat test
#chi2, p_value, dof, expectid

# Jointure des bases 

La **gravité** donne du sens aux fusions (merges) que je vais faire :

**Lieux** + Gravité : Quelles infrastructures sont les plus meurtrières ?

**Véhicules** + Gravité : Les airbags ou la taille du véhicule protègent-ils réellement ?

**Usagers** + Gravité : Quel est l'impact de l'âge ou de la position dans la voiture ?

**Caracteristique** + Gravité : quel est l'état réel de la voiture au moment de l'accident ?

In [127]:
print(df_caract_clean.shape)
print(df_lieux_clean.shape)
print(df_usag_clean.shape)
print(df_veh_clean.shape)

(56519, 14)
(56525, 18)
(56521, 15)
(97365, 10)


In [83]:
# Jointure des tables df_caract et  df_lieux
df_caract_lieux= df_caract_clean.merge(df_lieux_clean,left_on="Num_Acc",right_on="Num_Acc",how="inner")
df_caract_lieux.head()

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,...,vosp,prof,pr,pr1,plan,larrout,surf,infra,situ,vma
0,202100037733,28,5,2021,14:15,1,CC_97407,2,2,1,...,0,1,0,0,1,-1.0,1,5,1,50
1,202100049382,2,3,2021,10:59,1,CC_63113,2,1,1,...,0,1,0,0,1,-1.0,1,0,1,50
2,202100038891,20,5,2021,16:50,1,CC_88160,2,2,1,...,0,1,0,920,1,-1.0,1,0,1,50
3,202100028616,14,7,2021,19:45,1,CC_80021,2,1,1,...,0,4,-1,-1,1,-1.0,1,0,1,50
4,202100033509,17,6,2021,23:40,3,CC_13214,2,1,1,...,0,2,279,0,2,-1.0,1,0,1,70


In [47]:
df_caract_lieux.shape

(56526, 31)

In [64]:
df_caract_lieux[(df_caract_lieux["Num_Acc"].isna())]

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,...,vosp,prof,pr,pr1,plan,larrout,surf,infra,situ,vma


In [100]:
# Jointure des tables df_caract_lieux et df_usag_clean
df_caract_lieux_usag= df_caract_lieux.merge(df_usag_clean,left_on="Num_Acc",right_on="Num_Acc",how="inner")
df_caract_lieux_usag.head()

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,...,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202100037733,28,5,2021,14:15,1,CC_97407,2,2,1,...,4,1,1970,0,8,8,-1,-1,-1,-1
1,202100049382,2,3,2021,10:59,1,CC_63113,2,1,1,...,1,1,1979,1,1,0,-1,0,0,-1
2,202100038891,20,5,2021,16:50,1,CC_88160,2,2,1,...,1,1,1931,5,1,0,-1,-1,-1,-1
3,202100028616,14,7,2021,19:45,1,CC_80021,2,1,1,...,4,2,1973,0,1,-1,-1,0,0,-1
4,202100033509,17,6,2021,23:40,3,CC_13214,2,1,1,...,1,1,2002,5,1,0,-1,-1,-1,-1


In [121]:
df_caract_lieux_usag.shape

(56525, 45)

In [122]:
df_caract_lieux_usag[(df_caract_lieux_usag["Num_Acc"].isna())]

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,...,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp


In [116]:
# Affiche les données et le type (dtype) en bas de la liste
df_caract_lieux_usag['id_vehicule']

0        134A 220
1        113A 491
2        132A 115
3        150A 566
4        141A 884
           ...   
56520    153A 902
56521    166A 228
56522    172A 966
56523    130A 384
56524    118A 220
Name: id_vehicule, Length: 56525, dtype: object

In [115]:
df_veh['id_vehicule']

0        129A983
1        192A850
2        152A776
3        193A398
4        178A605
          ...   
97360    108A341
97361    200A693
97362    192A529
97363    174A733
97364    181A572
Name: id_vehicule, Length: 97365, dtype: object

In [118]:
# Nettoyage des espaces internes et mise en majuscules pour les deux tables
df_caract_lieux_usag['id_vehicule'] = df_caract_lieux_usag['id_vehicule'].astype(str).str.replace(r'\s+', '', regex=True).str.upper()
df_veh['id_vehicule'] = df_veh['id_vehicule'].astype(str).str.replace(r'\s+', '', regex=True).str.upper()

Le symbole **\s+** en expression régulière (**regex=True**) détecte tous les caractères "blancs" (espaces, tabulations, retours à la ligne) et les supprime.

In [124]:
# la jointure des 4 bases de données
df_final = df_caract_lieux_usag.merge(df_veh, on='id_vehicule', how='inner')

print(f"Nombre de lignes dans la base finale : {len(df_final)}")

Nombre de lignes dans la base finale : 56525


In [123]:
#Affichage de la jointure de la base finale
df_final

,Num_Acc,jour,mois,an,hrmn,lum,com,agg,int,atm,...,etatp,num_veh_y,senc,catv,obs,obsm,choc,manv,motor,occutc
0,202100037733,28,5,2021,14:15,1,CC_97407,2,2,1,...,-1,B01,3,1,0,2,8,1,5,0
1,202100049382,2,3,2021,10:59,1,CC_63113,2,1,1,...,-1,A01,1,10,0,2,1,2,1,0
2,202100038891,20,5,2021,16:50,1,CC_88160,2,2,1,...,-1,A01,1,7,0,2,1,1,1,0
3,202100028616,14,7,2021,19:45,1,CC_80021,2,1,1,...,-1,A01,0,7,0,2,1,15,1,0
4,202100033509,17,6,2021,23:40,3,CC_13214,2,1,1,...,-1,A01,2,7,4,5,1,26,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56520,202100026753,28,7,2021,17:10,1,CC_38474,2,2,1,...,-1,A01,2,1,0,2,2,0,5,0
56521,202100019814,10,9,2021,06:15,3,CC_77251,1,1,1,...,-1,A01,2,34,0,5,7,21,1,0
56522,202100016048,29,9,2021,08:15,1,CC_78524,1,1,1,...,-1,A01,2,33,0,2,3,3,1,0
56523,202100039830,15,5,2021,06:45,2,CC_13212,2,2,1,...,-1,A01,3,10,0,2,1,1,0,0


## Constituer les bases de train et test

**la préparation de ta variable cible**

Dans le jeu de données BAAC, la colonne grav (gravité) possède 4 modalités :

1 : Indemne

2 : Tué

3 : Blessé hospitalisé

4 : Blessé léger

Pour le modèle de **Machine Learning**, il est souvent préférable de simplifier cela en une classification binaire (Grave vs Non-Grave).

In [125]:
# Vérification de la distribution de la gravité
print("Répartition des classes de gravité :")
print(df_final['grav'].value_counts())

# Vérification du nombre total de lignes et colonnes
print(f"\nDimensions finales : {df_final.shape}")

Répartition des classes de gravité :
grav
4     23403
1     21267
3      9989
2      1848
-1       18
Name: count, dtype: Int64

Dimensions finales : (56525, 54)


**Création de la variable binaire**

Selon la nomenclature officielle, nous avons isoler les conséquences les plus lourdes :

- Grave (1) : Tués (2) et Blessés hospitalisés (3),

- Non-Grave (0) : Indemnes (1) et Blessés légers (4).

In [127]:
# Création de la fonction de mapping
def mapping_gravite(val):
    if val in [2, 3]: # Tué ou Blessé hospitalisé
        return 1
    else: # Indemne ou Blessé léger
        return 0

# Application sur une nouvelle colonne 'target'
df_final['target'] = df_final['grav'].apply(mapping_gravite)

# Vérification du nouvel équilibre
print("Répartition de la variable cible (Target) :")
print(df_final['target'].value_counts(normalize=True) * 100)

Répartition de la variable cible (Target) :
target
0    79.058824
1    20.941176
Name: proportion, dtype: float64


In [128]:
# On ne garde que les colonnes numériques ou catégorielles encodées
# On supprime les identifiants et les colonnes textuelles (comme 'adr')
cols_exclues = ['Num_Acc', 'id_vehicule', 'num_veh', 'adr', 'grav']
df_prep = df_final.drop(columns=[c for c in cols_exclues if c in df_final.columns])

# Gestion des valeurs manquantes (remplacement par le mode ou 0)
df_prep = df_prep.fillna(0) 

# Conversion des colonnes "object" en catégories (si nécessaire)
for col in df_prep.select_dtypes(include=['object']).columns:
    df_prep[col] = df_prep[col].astype('category').cat.codes

In [130]:


# X = les caractéristiques (météo, route, véhicule)
# y = la cible (target : 0 ou 1)
X = df_prep.drop(columns=['target'])
y = df_prep['target']

# Division 80% Train / 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Base d'entraînement : {X_train.shape[0]} accidents")
print(f"Base de test : {X_test.shape[0]} accidents")

Base d'entraînement : 45220 accidents
Base de test : 11305 accidents


In [131]:
# Sauvegarde pour ne pas perdre le split
X_train.to_csv('data/X_train.csv', index=False)
X_test.to_csv('data/X_test.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

print("Tes bases de Train et Test sont prêtes dans le dossier /data !")

Tes bases de Train et Test sont prêtes dans le dossier /data !
